# Restaurant Revenue Human Baseline Notebook

This notebook records a true bank-id-authored, official-only human baseline for `restaurant-revenue-prediction`. It intentionally picks one conservative date branch (`OpenDays`) and one compact low-cardinality encoding branch (`label`) rather than stacking mutually alternative bank rows.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

ROOT = Path("..").resolve()
DATA_DIR = ROOT / "data"
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
P_COLS = [f"P{i}" for i in range(1, 38)]
CAT_COLS = ["City Group", "Type"]


## Load Official Files

In [ ]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

train.shape, test.shape


## Drop Id

In [ ]:
train = train.drop(columns=["Id"])
test = test.drop(columns=["Id"])

"Id" in train.columns, "Id" in test.columns


## Parse Open Date

In [ ]:
train["Open Date"] = pd.to_datetime(train["Open Date"], format="%m/%d/%Y", errors="coerce")
test["Open Date"] = pd.to_datetime(test["Open Date"], format="%m/%d/%Y", errors="coerce")

train["Open Date"].dtype, test["Open Date"].dtype


## Derive OpenDays

In [ ]:
anchor_date = pd.Timestamp("2015-01-01")
train["OpenDays"] = (anchor_date - train["Open Date"]).dt.days
test["OpenDays"] = (anchor_date - test["Open Date"]).dt.days

train[["Open Date", "OpenDays"]].head()


## Label Encode Low-Cardinality Categoricals

In [ ]:
for column in CAT_COLS:
    values = sorted(pd.concat([train[column], test[column]], axis=0).dropna().unique())
    mapping = {value: idx for idx, value in enumerate(values)}
    train[column] = train[column].map(mapping).astype(int)
    test[column] = test[column].map(mapping).astype(int)

train[CAT_COLS].head()


## Add Istanbul Indicator

In [ ]:
train["City_Is_Istanbul"] = train["City"].eq("İstanbul").astype(int)
test["City_Is_Istanbul"] = test["City"].eq("İstanbul").astype(int)

train[["City", "City_Is_Istanbul"]].head()


## Standard Scale P-Features

In [ ]:
scaler = StandardScaler()
train[P_COLS] = scaler.fit_transform(train[P_COLS])
test[P_COLS] = scaler.transform(test[P_COLS])

train[P_COLS[:3]].head()


## Log1p Target

In [ ]:
target = np.log1p(train["revenue"])
target.head()


## Drop Raw Open Date

In [ ]:
train = train.drop(columns=["Open Date"])
test = test.drop(columns=["Open Date"])

"Open Date" in train.columns, "Open Date" in test.columns


## Feature Matrix Preview

In [ ]:
model_columns = P_COLS + ["City Group", "Type", "OpenDays", "City_Is_Istanbul"]
X_train = train[model_columns]
X_test = test[model_columns]

X_train.shape, X_test.shape, target.shape


## Scope Boundary

In [ ]:
excluded_steps = [
    "year/month/day date-part alternative",
    "one-hot alternative for City Group and Type",
    "row filtering on revenue or city subsets",
    "external geocoding, holiday calendars, or city priors",
]
excluded_steps
